In [2]:
import pandas as pd
import numpy as np

In [3]:
df=pd.read_excel(r"C:\Users\samru\OneDrive\Desktop\SIH_2025\AI-Pollution-Forecast-and-Policy-Dashboard\ML\Raw\CCR_DATA_2023_25\2025_Ashok Vihar, Delhi - DPCC.xlsx",skiprows=16)

In [4]:
# ---------- 2. Remove duplicate rows and columns ----------
df = df.drop_duplicates().reset_index(drop=True)
df = df.loc[:, ~df.T.duplicated()]
print("Shape after removing duplicates:", df.shape)

Shape after removing duplicates: (320, 20)


In [5]:
# ---------- 3. Handle missing values (drop >70% NaN, impute median otherwise) ----------
nan_thresh = 0.7

# Drop columns with >70% missing
cols_to_drop = df.columns[df.isnull().mean() > nan_thresh]
df = df.drop(columns=cols_to_drop)
print(f"Dropped columns (>{int(nan_thresh*100)}% NaN): {cols_to_drop.tolist()}")

# Drop rows with >70% missing
rows_to_drop = df.index[df.isnull().mean(axis=1) > nan_thresh]
df = df.drop(index=rows_to_drop).reset_index(drop=True)
print(f"Dropped rows (>{int(nan_thresh*100)}% NaN):", len(rows_to_drop))

# Impute remaining missing values
num_cols = df.select_dtypes(include=[np.number]).columns
cat_cols = df.select_dtypes(exclude=[np.number]).columns

for col in num_cols:
    median_val = df[col].median()
    df[col] = df[col].fillna(median_val)

for col in cat_cols:
    if df[col].isnull().any():
        mode_val = df[col].mode(dropna=True)
        if not mode_val.empty:
            df[col] = df[col].fillna(mode_val[0])

print("Missing values after imputation:\n", df.isnull().sum())

Dropped columns (>70% NaN): []
Dropped rows (>70% NaN): 0
Missing values after imputation:
 From Date    0
To Date      0
PM2.5        0
PM10         0
NO           0
NO2          0
NOx          0
NH3          0
SO2          0
CO           0
Ozone        0
Benzene      0
Toluene      0
RH           0
WS           0
WD           0
BP           0
AT           0
RF           0
TOT-RF       0
dtype: int64


In [6]:
# ---------- 4. Handle outliers using IQR (with 70% rule) ----------

def get_outlier_mask(series):
    q1 = series.quantile(0.25)
    q3 = series.quantile(0.75)
    iqr = q3 - q1
    lower = q1 - 1.5 * iqr
    upper = q3 + 1.5 * iqr
    return (series < lower) | (series > upper)

# Handle columns: drop if >70% outliers, else replace with median
outlier_thresh = 0.7
cols_to_drop = []
for col in num_cols:
    outlier_mask = get_outlier_mask(df[col])
    outlier_fraction = outlier_mask.mean()
    if outlier_fraction > outlier_thresh:
        cols_to_drop.append(col)
    else:
        median_val = df[col].median()
        df.loc[outlier_mask, col] = median_val
if cols_to_drop:
    df = df.drop(columns=cols_to_drop)
    print(f"Dropped numeric columns (>{int(outlier_thresh*100)}% outliers): {cols_to_drop}")

# Handle rows: drop if >70% numeric columns are outliers in a given row
outlier_matrix = df[num_cols].apply(get_outlier_mask)
row_outlier_fraction = outlier_matrix.mean(axis=1)
rows_to_drop = df.index[row_outlier_fraction > outlier_thresh]
df = df.drop(index=rows_to_drop).reset_index(drop=True)
if len(rows_to_drop) > 0:
    print(f"Dropped rows (>{int(outlier_thresh*100)}% outliers): {len(rows_to_drop)}")

In [7]:
# ---------- 5. Convert date columns to datetime ----------
if 'From Date' in df.columns:
    df['From Date'] = pd.to_datetime(df['From Date'], errors='coerce')
if 'To Date' in df.columns:
    df['To Date'] = pd.to_datetime(df['To Date'], errors='coerce')

In [8]:
# ---------- 6. Final check ----------
print("Final shape:", df.shape)
print(df.head())

Final shape: (320, 20)
   From Date    To Date   PM2.5    PM10     NO    NO2    NOx    NH3    SO2  \
0 2025-01-01 2025-02-01  167.83  274.01   6.39  43.59  26.92  41.05  11.40   
1 2025-02-01 2025-03-01  193.50  305.79  17.28  41.47  32.39  63.57  10.91   
2 2025-03-01 2025-04-01   68.02  451.50   6.23  61.85  24.43  31.68  12.16   
3 2025-04-01 2025-05-01   68.02  447.73   6.23  73.51  24.43  31.68  12.00   
4 2025-05-01 2025-06-01  187.75  291.54  12.65  51.04  37.42  58.51   9.27   

     CO  Ozone  Benzene  Toluene     RH    WS     WD      BP     AT   RF  \
0  0.97  22.21     0.63     2.67  83.67  1.32  57.63  981.91  12.62  0.0   
1  1.58  19.66     0.78     3.21  85.36  1.36  58.21  982.00  12.68  0.0   
2  0.97  15.88     1.25     7.14  86.88  1.20  56.49  981.93  13.39  0.0   
3  1.67  12.05     1.53     7.99  88.16  1.09  59.47  981.91  13.58  0.0   
4  1.48  20.28     0.79     2.42  83.02  1.23  58.67  981.97  13.61  0.0   

   TOT-RF  
0     0.0  
1     0.0  
2     0.0  
3  

In [9]:
from sklearn.preprocessing import StandardScaler
scaler = StandardScaler()
numerics = df.select_dtypes(include=[np.number]).columns
df[numerics] = scaler.fit_transform(df[numerics])


In [27]:
df

,From Date,To Date,PM2.5,PM10,NO,NO2,NOx,NH3,SO2,CO,Ozone,Benzene,Toluene,RH,WS,WD,BP,AT,RF,TOT-RF
0,2025-01-01,2025-02-01,1.887407,0.788072,-0.187156,0.094064,-0.059011,0.741949,-0.652700,-0.117890,-1.188176,0.787394,0.043025,1.341710,-0.182440,-0.947992,0.657186,-2.226446,0.0,0.0
1,2025-02-01,2025-03-01,2.421739,1.095652,2.187164,-0.015985,0.351503,2.262523,-0.758582,1.298243,-1.304481,1.274756,0.294456,1.462623,-0.082515,-0.939078,0.828797,-2.216857,0.0,0.0
2,2025-03-01,2025-04-01,-0.190182,2.505894,-0.222041,1.041937,-0.245882,0.109277,-0.488475,-0.117890,-1.476885,2.801823,2.124312,1.571373,-0.482212,-0.965512,0.695322,-2.103379,0.0,0.0
3,2025-04-01,2025-05-01,-0.190182,2.469406,-0.222041,1.647206,-0.245882,0.109277,-0.523048,1.507181,-1.651570,3.711565,2.520083,1.662952,-0.757003,-0.919714,0.657186,-2.073011,0.0,0.0
4,2025-05-01,2025-06-01,2.302051,0.957735,1.177696,0.480792,0.728996,1.920867,-1.112962,1.066090,-1.276203,1.307246,-0.073378,1.295205,-0.407269,-0.932009,0.771593,-2.068216,0.0,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
315,2025-12-11,NaT,-0.190182,-0.084921,3.224975,2.509428,2.862618,-1.770510,1.140810,2.505438,-0.160592,1.534682,1.723886,-0.412599,-0.207421,1.281980,0.828797,-1.133222,0.0,0.0
316,NaT,NaT,-0.190182,2.926905,0.628267,-0.241014,2.605953,-1.754305,0.432049,1.832195,-0.434250,2.444424,2.035846,-0.243751,-1.006814,1.393094,0.828797,-1.185965,0.0,0.0
317,NaT,NaT,-0.190182,2.200249,0.373175,2.902904,2.368800,-1.742151,0.695674,1.275028,-0.426496,1.307246,1.239649,-0.182937,-0.981833,1.402162,0.828797,-1.237110,0.0,0.0
318,NaT,NaT,-0.190182,2.375235,1.079584,2.319957,1.917010,-1.781989,0.483910,0.903583,-0.561957,1.599664,0.788005,-0.288109,-1.106738,1.499137,0.828797,-1.229119,0.0,0.0


In [11]:
df.to_excel('Ashokvihar2025.xlsx', index=False)